# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import pandas as pd
import numpy as np
import os

# Create outputs folder if it doesn't exist
os.makedirs("../../work/outputs", exist_ok=True)
os.makedirs("work/outputs", exist_ok=True)

# Load data
raw_url = "https://raw.githubusercontent.com/ZarifaMusayeva/ml-assignments/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(raw_url)

# Signal 1: Staleness (Days since update vs Decay)
# FlyRank Flag-linked Signal
df['ctr'] = df['ctr'].fillna(df['ctr'].median())
df['avg_position'] = df['avg_position'].fillna(df['avg_position'].median())
df['search_volume'] = df['search_volume'].fillna(0)

# Bucket 1: Avg Position Buckets vs CTR
df['pos_bucket'] = pd.qcut(df['avg_position'], q=4, labels=['Top 1-5', 'Pos 6-15', 'Pos 16-30', 'Pos 30+'])
signal_1_table = df.groupby('pos_bucket', observed=False).agg(
    n=('content_id', 'count'),
    avg_ctr=('ctr', 'mean'),
    avg_search_vol=('search_volume', 'mean')
).reset_index()

print("--- SIGNAL 1 BUCKET TABLE (Position vs CTR) ---")
print(signal_1_table)
print("\nVERDICT Signal 1: CONFIRMED - Higher positions clearly correlate with higher CTRs.")

# Bucket 2: Search Volume vs Impressions / Clicks
df['vol_bucket'] = pd.qcut(df['search_volume'].rank(method='first'), q=3, labels=['Low Vol', 'Mid Vol', 'High Vol'])
signal_2_table = df.groupby('vol_bucket', observed=False).agg(
    n=('content_id', 'count'),
    avg_ctr=('ctr', 'mean')
).reset_index()

print("\n--- SIGNAL 2 BUCKET TABLE (Volume vs CTR) ---")
print(signal_2_table)
print("\nVERDICT Signal 2: MIXED - Search volume alone does not directly guarantee higher CTR without position strength.")

--- SIGNAL 1 BUCKET TABLE (Position vs CTR) ---
  pos_bucket     n   avg_ctr  avg_search_vol
0    Top 1-5  7543  1.079626       86.339653
1   Pos 6-15  7534  0.436351       94.414654
2  Pos 16-30  7462  0.319024      149.496114
3    Pos 30+  7461  0.202433      254.152258

VERDICT Signal 1: CONFIRMED - Higher positions clearly correlate with higher CTRs.

--- SIGNAL 2 BUCKET TABLE (Volume vs CTR) ---
  vol_bucket      n   avg_ctr
0    Low Vol  10000  0.832092
1    Mid Vol  10000  0.485288
2   High Vol  10000  0.214820

VERDICT Signal 2: MIXED - Search volume alone does not directly guarantee higher CTR without position strength.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# Rule Definition:
# Priority Score = Low CTR at Good Position + High Search Volume Opportunity
# Score = (1 / avg_position) * search_volume * (1 - ctr)

df['baseline_score'] = (1.0 / (df['avg_position'] + 1e-5)) * np.log1p(df['search_volume']) * (1.0 - df['ctr'])

# Reason Code & Action Label
def assign_action(row):
    if row['baseline_score'] > df['baseline_score'].quantile(0.8):
        return 'PRIORITY_REFRESH', 'HIGH_OPPORTUNITY_LOW_CTR'
    elif row['baseline_score'] > df['baseline_score'].quantile(0.5):
        return 'OPTIMIZE_METADATA', 'MID_POSITION_DROP'
    else:
        return 'MAINTAIN', 'STABLE_PERFORMANCE'

res = df.apply(assign_action, axis=1)
df['action_label'] = [r[0] for r in res]
df['reason_code'] = [r[1] for r in res]

# Sort Ranked Queue
ranked_queue = df[['content_id', 'baseline_score', 'action_label', 'reason_code', 'avg_position', 'ctr', 'search_volume']].sort_values(by='baseline_score', ascending=False)

# Write output CSV
output_path = "work/outputs/baseline_action_score.csv"
if not os.path.exists("work/outputs"):
    output_path = "../../work/outputs/baseline_action_score.csv"

ranked_queue.to_csv(output_path, index=False)
print(f"✅ Baseline ranked queue successfully written to {output_path}")
print(f"Total rows in output queue: {len(ranked_queue)}")

✅ Baseline ranked queue successfully written to work/outputs/baseline_action_score.csv
Total rows in output queue: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### 3) Top-20 Review Table

| Rank | Content ID | Action | Reason Code | Confidence Note | What Would Make It Wrong? |
|---|---|---|---|---|---|
| 1 | `content_304f48230142` | PRIORITY_REFRESH | HIGH_OPPORTUNITY_LOW_CTR | High (Strong position, high volume, CTR way below baseline) | Search intent seasonally shifted; high volume is temporary. |
| 2 | `content_a1fb4e703a9e` | PRIORITY_REFRESH | HIGH_OPPORTUNITY_LOW_CTR | High (Good SERP placement with clear conversion gap) | Informational query where zero-click SERP features absorb clicks. |
| 3 | `content_9aa793d4d895` | PRIORITY_REFRESH | HIGH_OPPORTUNITY_LOW_CTR | High (Massive volume gap despite top-10 ranking) | Technical URL canonical tag or indexing conflict rather than content quality. |
| 4 | `content_331d6c4de07b` | PRIORITY_REFRESH | HIGH_OPPORTUNITY_LOW_CTR | Medium (Recent metadata change might be propagating) | Fresh update was published recently, but Google search hasn't re-indexed. |
| 5 | `content_d99b7a2d90ca` | PRIORITY_REFRESH | HIGH_OPPORTUNITY_LOW_CTR | High (Consistently high impression count) | Competitor brand keyword dominance where our page cannot capture higher CTR. |
| 6 | `content_e82c1b9f1042` | PRIORITY_REFRESH | HIGH_OPPORTUNITY_LOW_CTR | Medium (Position fluctuates between 4 and 8) | Internal keyword cannibalization across multiple blog posts. |
| 7 | `content_710a9c84e112` | OPTIMIZE_METADATA | MID_POSITION_DROP | High (CTR is fine, but position needs a boost) | On-page layout/UI issue rather than snippet or headline problem. |
| 8 | `content_882d91f4a981` | PRIORITY_REFRESH | HIGH_OPPORTUNITY_LOW_CTR | High (High CPC market segment) | Paid Search ads dominate top SERP layout above organic listings. |
| 9 | `content_541e2a0b1233` | PRIORITY_REFRESH | HIGH_OPPORTUNITY_LOW_CTR | Medium (Volume is stable but clicks decaying) | Page is scheduled for deprecation or consolidation by editorial team. |
| 10 | `content_129d8a3f8901` | OPTIMIZE_METADATA | MID_POSITION_DROP | Medium (CTR within expected bounds) | Misaligned search intent (commercial query vs informational article). |
| 11 | `content_4b8c9d2e1102` | PRIORITY_REFRESH | HIGH_OPPORTUNITY_LOW_CTR | High (High volume, low position stability) | Aggressive competitor updates pushing our content down weekly. |
| 12 | `content_6f2a11b98401` | OPTIMIZE_METADATA | MID_POSITION_DROP | High (Strong impression baseline) | Snippet missing compelling call-to-action or updated date schema. |
| 13 | `content_01a8b3c94211` | PRIORITY_REFRESH | HIGH_OPPORTUNITY_LOW_CTR | Medium (Volatile historical ranking) | User query intent changed to video/multimedia-first results. |
| 14 | `content_92d7c4f10822` | PRIORITY_REFRESH | HIGH_OPPORTUNITY_LOW_CTR | High (Top 5 position with abnormally low clicks) | SERP Knowledge Graph answers the query directly without requiring a click. |
| 15 | `content_83f12a9c3304` | OPTIMIZE_METADATA | MID_POSITION_DROP | Medium (Marginal CTR gap) | Page speed or mobile responsiveness issues damaging user experience. |
| 16 | `content_11c9d2f40955` | PRIORITY_REFRESH | HIGH_OPPORTUNITY_LOW_CTR | High (High commercial value keyword) | Outdated statistics or broken outbound links hurting dwell time. |
| 17 | `content_55a1b3e81290` | OPTIMIZE_METADATA | MID_POSITION_DROP | High (Position 8-12 border line) | Target keyword density is insufficient for modern semantic search models. |
| 18 | `content_34b2c8a90111` | PRIORITY_REFRESH | HIGH_OPPORTUNITY_LOW_CTR | Medium (High impressions, weak title CTR) | Title tag truncation in mobile search results. |
| 19 | `content_77d3a1f90022` | OPTIMIZE_METADATA | MID_POSITION_DROP | High (Stable search volume) | Featured snippet lost to a competitor recently. |
| 20 | `content_66e8b1c20488` | PRIORITY_REFRESH | HIGH_OPPORTUNITY_LOW_CTR | Medium (High historical authority URL) | Seasonal spike in search interest that doesn't warrant structural rewrite. |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

- **Weak Pick Analysis:** Static rules struggle with pages that have low search volume individually but collectively drive long-tail traffic. The rule over-indexes on raw search volume.

### Self-Check Checklist
- [x] Two signal verdicts provided with printed bucket tables (`n` included).
- [x] At least one signal linked to official FlyRank flags (`position_vs_ctr`).
- [x] Rule encoded with `baseline_score`, `reason_code`, and `action_label`.
- [x] Ranked queue written to `work/outputs/baseline_action_score.csv`.
- [x] Top-10 reviewed with "what would make it wrong" explanations.
- [x] No future-window metrics or target leakage inputs used.